# 페트병 4클래스 분류 모델 학습 (YOLO11n-cls)

전처리 담당이 만든 `pet4_split` 폴더(train/val, 클래스 4개)로 분류 모델을 학습합니다.

| 단계 | 내용 |
| --- | --- |
| 0 | Colab 런타임을 **GPU**로 바꾸기 (런타임 → 런타임 유형 변경 → T4 GPU) |
| 1 | 드라이브 연결, 데이터 복사 |
| 2 | 데이터 확인 |
| 3 | 실험 A: 그대로 학습 |
| 4 | 실험 B: 클래스 균형 맞춰 학습 |
| 5 | 평가: 클래스별 재현율, 혼동행렬, 3개 판정(적합·라벨불량·기타)으로 합친 결과 |
| 6 | 틀린 사진 모아 보기, 신뢰도 기준 점검 |
| 7 | 결과를 드라이브에 저장 |
| 8 | 웹캠 판정용 함수 (레터박스 + 3프레임 다수결) |

**먼저 아래 설정 칸의 경로만 바꾸면** 나머지는 위에서부터 차례로 실행하면 됩니다.

In [ ]:
# ===== 설정: 여기만 바꾸세요 =====
DRIVE_ZIP   = "/content/drive/MyDrive/pet/pet4_split.zip"   # 드라이브에 올린 pet4_split 압축 파일
DRIVE_SAVE  = "/content/drive/MyDrive/pet/runs"             # 학습 결과를 저장할 드라이브 폴더
WORK        = "/content/work"                               # Colab 로컬 작업 폴더 (빠름, 세션 끝나면 사라짐)

CLASSES = ["clear_nolabel", "clear_label", "color_nolabel", "color_label"]

# 4클래스 -> 실제 판정 3개로 합치는 규칙
DECISION = {
    "clear_nolabel": "적합",      # 투명 페트 통
    "clear_label":   "라벨불량",  # 날개판으로 배출
    "color_nolabel": "기타",      # 기타 플라스틱
    "color_label":   "기타",
}

CONF_THRESHOLD = 0.6   # 이 값보다 신뢰도가 낮으면 '재검사'
IMGSZ  = 224
EPOCHS = 60
SEED   = 0

## 1. 준비: 패키지 설치, 드라이브 연결, 데이터 복사

In [ ]:
!pip -q install ultralytics scikit-learn

from google.colab import drive
drive.mount("/content/drive")

import os, zipfile, shutil, glob
os.makedirs(WORK, exist_ok=True)
with zipfile.ZipFile(DRIVE_ZIP) as z:
    z.extractall(WORK)

# 압축 안에 pet4_split 폴더가 한 겹 더 있을 수 있어 train 폴더 위치로 찾습니다
cands = [os.path.dirname(p) for p in glob.glob(f"{WORK}/**/train", recursive=True)]
assert cands, "train 폴더를 찾지 못했습니다. 압축 파일 구조를 확인하세요."
DATA = sorted(cands, key=len)[0]
print("데이터 폴더:", DATA)

import torch
print("GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "없음 (CPU로 학습, 느림)")

## 2. 데이터 확인
클래스별 장수와 샘플 사진을 봅니다. 숫자가 전처리 문서와 같은지 확인하세요.

In [ ]:
import pandas as pd
IMG_EXT = (".jpg", ".jpeg", ".png", ".bmp", ".webp")

def count_images(root):
    rows = []
    for split in ["train", "val"]:
        for c in CLASSES:
            d = os.path.join(root, split, c)
            n = len([f for f in os.listdir(d) if f.lower().endswith(IMG_EXT)]) if os.path.isdir(d) else 0
            rows.append({"split": split, "class": c, "n": n})
    return pd.DataFrame(rows).pivot(index="class", columns="split", values="n").loc[CLASSES]

counts = count_images(DATA)
print(counts)

# 폴더 이름이 CLASSES와 다르면 여기서 멈춥니다
for split in ["train", "val"]:
    found = sorted(d for d in os.listdir(os.path.join(DATA, split)) if os.path.isdir(os.path.join(DATA, split, d)))
    assert set(found) == set(CLASSES), f"{split} 폴더의 클래스가 다릅니다: {found}"

In [ ]:
import matplotlib.pyplot as plt
from PIL import Image
import random
random.seed(SEED)

fig, axes = plt.subplots(len(CLASSES), 6, figsize=(12, 2.2 * len(CLASSES)))
for r, c in enumerate(CLASSES):
    files = glob.glob(os.path.join(DATA, "train", c, "*"))
    for k, f in enumerate(random.sample(files, min(6, len(files)))):
        axes[r, k].imshow(Image.open(f).convert("RGB")); axes[r, k].axis("off")
    axes[r, 0].set_title(c, loc="left", fontsize=10)
plt.tight_layout(); plt.show()

## 3. 학습 함수

증강 설정 이유:

- `hsv_h=0` 색조 변경 끔: 색 자체가 정답이라 투명/유색이 바뀌면 안 됩니다.
- `auto_augment=None` RandAugment 끔: 채도·대비·색 보정 연산이 섞여 있습니다. (끄면 아래 hsv 값이 적용됩니다)
- `erasing=0` 무작위 지우기 끔: 라벨 부분이 지워지면 정답이 바뀝니다.
- `scale=0.2` 무작위 자르기를 약하게(원본의 80~100%): 라벨이 잘려 나가지 않게 합니다.
- `fliplr`, `flipud` 반전: 벨트 위 병은 방향이 제각각이라 괜찮습니다.

학습 후 결과 폴더의 `train_batch*.jpg`를 열어 증강된 사진에서 라벨이 잘리거나 색이 바뀌지 않았는지 꼭 확인하세요.

In [ ]:
from ultralytics import YOLO

def train(data_dir, name, epochs=EPOCHS):
    model = YOLO("yolo11n-cls.pt")          # ImageNet 사전학습 가중치에서 시작
    model.train(
        data=data_dir, imgsz=IMGSZ, epochs=epochs, patience=15, batch=64,
        seed=SEED, deterministic=True,
        hsv_h=0.0, hsv_s=0.1, hsv_v=0.3,
        auto_augment=None, erasing=0.0, scale=0.2,
        fliplr=0.5, flipud=0.5,
        project=f"{WORK}/runs", name=name, exist_ok=True,
        plots=True,
    )
    best = f"{WORK}/runs/{name}/weights/best.pt"
    print("best 모델:", best)
    return best

## 4. 실험 A: 데이터 그대로

In [ ]:
best_A = train(DATA, "A_baseline")

## 5. 실험 B: 클래스 균형

학습용(train)만 바꿉니다. **평가용(val)은 그대로 둡니다.**

- 많은 클래스(color_nolabel)는 `CAP`장까지만 무작위로 남깁니다.
- 적은 클래스는 `MIN_N`장이 될 때까지 같은 사진을 복사해 넣습니다. 복사본은 학습 때마다 다르게 증강됩니다.

In [ ]:
CAP, MIN_N = 300, 200
random.seed(SEED)

DATA_B = f"{WORK}/pet4_balanced"
if os.path.exists(DATA_B): shutil.rmtree(DATA_B)
shutil.copytree(os.path.join(DATA, "val"), os.path.join(DATA_B, "val"))   # val은 그대로 복사

for c in CLASSES:
    src = sorted(glob.glob(os.path.join(DATA, "train", c, "*")))
    dst = os.path.join(DATA_B, "train", c); os.makedirs(dst)
    pick = random.sample(src, CAP) if len(src) > CAP else list(src)
    for f in pick:
        shutil.copy(f, dst)
    k = 0
    while len(os.listdir(dst)) < MIN_N:        # 부족하면 복사해서 채움
        f = pick[k % len(pick)]
        base, ext = os.path.splitext(os.path.basename(f))
        shutil.copy(f, os.path.join(dst, f"{base}__dup{k}{ext}"))
        k += 1

print(count_images(DATA_B))

In [ ]:
best_B = train(DATA_B, "B_balanced")

## 6. 평가

`model.val()`은 전체 정확도만 알려 주므로, 평가용 사진을 하나씩 예측해서 직접 계산합니다.

- **클래스별 재현율**: KPI N5(클래스당 90% 이상)에 쓰는 숫자
- **3개 판정으로 합친 결과**: 실제로 병이 어디로 가는지 기준
- **가장 나쁜 실수**: 라벨 있는 투명 병(clear_label)을 적합(clear_nolabel)으로 통과시킨 수

In [ ]:
import numpy as np
from sklearn.metrics import confusion_matrix, classification_report

def predict_folder(weights, root):
    model = YOLO(weights)
    names = model.names                    # {0: 'clear_label', ...} 폴더 이름 순서
    rows = []
    for c in CLASSES:
        files = sorted(f for f in glob.glob(os.path.join(root, c, "*")) if f.lower().endswith(IMG_EXT))
        for i in range(0, len(files), 64):
            for f, r in zip(files[i:i+64], model.predict(files[i:i+64], imgsz=IMGSZ, verbose=False)):
                p = r.probs
                rows.append({"file": f, "true": c, "pred": names[int(p.top1)], "conf": float(p.top1conf)})
    return pd.DataFrame(rows)

def report(df, title):
    print("=" * 60, "\n", title)
    print("\n[4클래스]")
    print(classification_report(df.true, df.pred, labels=CLASSES, digits=3, zero_division=0))
    cm = pd.DataFrame(confusion_matrix(df.true, df.pred, labels=CLASSES),
                      index=[f"실제:{c}" for c in CLASSES], columns=[f"예측:{c}" for c in CLASSES])
    print(cm)

    t3, p3 = df.true.map(DECISION), df.pred.map(DECISION)
    labs3 = ["적합", "라벨불량", "기타"]
    print("\n[3개 판정으로 합친 결과]")
    print(classification_report(t3, p3, labels=labs3, digits=3, zero_division=0))

    worst = ((df.true == "clear_label") & (df.pred == "clear_nolabel")).sum()
    clear_to_color = (df.true.str.startswith("clear") & df.pred.str.startswith("color")).sum()
    low = (df.conf < CONF_THRESHOLD).mean()
    acc_hi = (df[df.conf >= CONF_THRESHOLD].true == df[df.conf >= CONF_THRESHOLD].pred).mean()
    print(f"라벨 병을 적합으로 통과: {worst}장 / clear_label {(df.true=='clear_label').sum()}장")
    print(f"투명 병을 유색으로 판정: {clear_to_color}장")
    print(f"신뢰도 {CONF_THRESHOLD} 미만(재검사) 비율: {low:.1%}  |  그 외 정확도: {acc_hi:.1%}")
    return cm

VAL = os.path.join(DATA, "val")
results = {}
for name, w in [("A_baseline", best_A), ("B_balanced", best_B)]:
    results[name] = predict_folder(w, VAL)
    report(results[name], name)

### 6-1. 직접 촬영 사진만 따로 평가 (선택)

평가용 사진 대부분이 온라인 사진이라, 실제 환경과 가까운 직접 촬영분만 따로 봅니다.
`split_manifest.csv`의 열 이름을 먼저 확인하고, 아래 두 줄을 맞게 고치세요. 장수가 적어 참고용입니다.

In [ ]:
man_path = glob.glob(f"{WORK}/**/split_manifest.csv", recursive=True)
if man_path:
    man = pd.read_csv(man_path[0])
    print(man.columns.tolist()); display(man.head())

    FILE_COL   = "file"     # ← 파일명(또는 경로)이 들어 있는 열 이름으로 바꾸세요
    SOURCE_COL = "source"   # ← 온라인/직접촬영 구분 열 이름으로 바꾸세요
    PHONE_VALUE = "phone"   # ← 직접촬영을 뜻하는 값으로 바꾸세요

    if FILE_COL in man.columns and SOURCE_COL in man.columns:
        phone = set(man.loc[man[SOURCE_COL].astype(str).str.contains(PHONE_VALUE), FILE_COL]
                    .map(lambda s: os.path.basename(str(s))))
        for name, df in results.items():
            sub = df[df.file.map(os.path.basename).isin(phone)]
            print(f"\n{name}: 직접 촬영 {len(sub)}장, 정확도 {(sub.true == sub.pred).mean():.1%}")
            print(pd.crosstab(sub.true, sub.pred))
    else:
        print("열 이름을 위에서 맞게 고친 뒤 다시 실행하세요.")
else:
    print("split_manifest.csv를 찾지 못했습니다.")

## 7. 틀린 사진 모아 보기
틀린 사진에서 반복되는 패턴(연한 색 병, 작은 라벨, 반사 등)을 찾으면 다음에 모을 데이터가 정해집니다.

In [ ]:
def show_wrong(df, n=24):
    wrong = df[df.true != df.pred].sort_values("conf", ascending=False).head(n)
    if wrong.empty:
        print("틀린 사진이 없습니다."); return
    cols = 6; rows = int(np.ceil(len(wrong) / cols))
    fig, axes = plt.subplots(rows, cols, figsize=(14, 2.6 * rows))
    for ax in np.array(axes).ravel(): ax.axis("off")
    for ax, (_, r) in zip(np.array(axes).ravel(), wrong.iterrows()):
        ax.imshow(Image.open(r.file).convert("RGB"))
        ax.set_title(f"true {r.true}\npred {r.pred} {r.conf:.2f}", fontsize=8)  # 그래프는 영어로 (Colab 기본 글꼴에 한글 없음)
    plt.tight_layout(); plt.show()

show_wrong(results["B_balanced"])   # A도 보려면 "A_baseline"

## 8. 결과를 드라이브에 저장
세션이 끊기면 `/content`가 사라지므로 학습이 끝나면 바로 실행하세요.

In [ ]:
os.makedirs(DRIVE_SAVE, exist_ok=True)
for name in results:
    shutil.copytree(f"{WORK}/runs/{name}", os.path.join(DRIVE_SAVE, name), dirs_exist_ok=True)
    results[name].to_csv(os.path.join(DRIVE_SAVE, name, "val_predictions.csv"), index=False, encoding="utf-8-sig")
print("저장 완료:", DRIVE_SAVE)

## 9. 웹캠 판정용 함수

판정할 때도 **학습 사진과 똑같이** 판정 구역을 자르고 회색 레터박스를 씌워야 합니다.

- `LETTERBOX_COLOR`는 전처리 담당이 쓴 회색 값과 **같게** 맞추세요. (예: 114 또는 128)
- `ROI`는 웹캠 화면에서 병이 멈추는 판정 구역 좌표입니다. 벨트 설치 후 정합니다.
- 연속 3프레임이 같은 클래스이고 신뢰도가 기준 이상일 때만 확정, 아니면 '재검사'.

In [ ]:
from collections import Counter

LETTERBOX_COLOR = (114, 114, 114)   # ← 전처리 코드의 회색 값과 같게
ROI = None                          # ← 예: (x1, y1, x2, y2). None이면 화면 전체

def letterbox(img, size=IMGSZ, color=LETTERBOX_COLOR):
    """비율을 유지해 size×size에 맞추고 남는 곳을 회색으로 채움 (PIL 이미지)"""
    img = img.convert("RGB")
    w, h = img.size; s = size / max(w, h)
    img = img.resize((max(1, round(w * s)), max(1, round(h * s))), Image.BILINEAR)
    canvas = Image.new("RGB", (size, size), color)
    canvas.paste(img, ((size - img.width) // 2, (size - img.height) // 2))
    return canvas

def classify_frame(model, frame_rgb):
    """frame_rgb: 웹캠 프레임(numpy RGB). OpenCV로 읽었다면 BGR→RGB 변환 후 넣기"""
    img = Image.fromarray(frame_rgb)
    if ROI: img = img.crop(ROI)
    r = model.predict(letterbox(img), imgsz=IMGSZ, verbose=False)[0]
    return model.names[int(r.probs.top1)], float(r.probs.top1conf)

def decide(preds):
    """preds: 연속 프레임의 (클래스, 신뢰도) 목록, 보통 3개"""
    classes = [c for c, _ in preds]
    top, cnt = Counter(classes).most_common(1)[0]
    if cnt == len(preds) and min(conf for _, conf in preds) >= CONF_THRESHOLD:
        return top, DECISION[top]
    return top, "재검사"

# 사용 예 (평가용 사진 3장으로 흉내)
model = YOLO(best_B)
demo = sorted(glob.glob(os.path.join(VAL, "clear_label", "*")))[:3]
preds = [classify_frame(model, np.array(Image.open(f).convert("RGB"))) for f in demo]
print(preds, "→", decide(preds))